# Waste Classification using MobileNetV3Large
Four classes: **Glass, Paper, Metal, Plastic**.

Dataset structure: `dataset/Glass`, `dataset/Paper`, `dataset/Metal`, `dataset/Plastic`.

In [ ]:
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large

print('TensorFlow:', tf.__version__)

In [ ]:
DATA_DIR = Path('dataset')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')

class_names = train_ds.class_names
print('Classes:', class_names)
Path('models').mkdir(exist_ok=True)
with open('models/class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=2)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
], name='augmentation')

In [ ]:
base_model = MobileNetV3Large(
    input_shape=(224, 224, 3), include_top=False,
    weights='imagenet', pooling='avg', include_preprocessing=True)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint('models/waste_mobilenetv3.keras', monitor='val_accuracy', save_best_only=True)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.show()

plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print(f'Validation loss: {loss:.4f}')
print(f'Validation accuracy: {accuracy:.4f}')
print('Model saved to models/waste_mobilenetv3.keras')

In [ ]:
def predict_image(image_path):
    image = Image.open(image_path).convert('RGB').resize(IMG_SIZE)
    arr = np.expand_dims(np.asarray(image, dtype=np.float32), axis=0)
    probs = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))
    print(f'Prediction: {class_names[idx]}')
    print(f'Confidence: {probs[idx]*100:.2f}%')

# predict_image('test.jpg')